In [1]:
#input necessary functions
import os

import csv
import matplotlib.pyplot as plt
import numpy as np
import numpy #todo-redundant import
import pandas as pd
import random
import shutil
import torch
import xml.etree.ElementTree as ET

# Acquisition function
from botorch.acquisition import qLogExpectedImprovement
from botorch.sampling.stochastic_samplers import StochasticSampler
from botorch.utils.transforms import unnormalize
from botorch.optim import optimize_acqf

from skopt import gp_minimize
from skopt.plots import plot_convergence, plot_objective, plot_evaluations
from skopt.space import Real
from skopt.utils import use_named_args

#Sobol sequence generate initial locations
from torch.quasirandom import SobolEngine
from botorch.utils.transforms import unnormalize

### GP model surrogate
import gpytorch # todo:do we need full package import?
from botorch import fit_gpytorch_mll
from gpytorch.likelihoods import GaussianLikelihood
from gpytorch.mlls import ExactMarginalLogLikelihood
from gpytorch.means.mean import Mean
from botorch.models import SingleTaskGP
from botorch.models.transforms import Standardize
from botorch.utils.transforms import normalize
from gpytorch.constraints import Interval
from gpytorch.kernels import MaternKernel, ScaleKernel, RBFKernel, Kernel, rbf_kernel

# we may no longer  need these
import math
from gpytorch.functions import MaternCovariance
from gpytorch.settings import trace_mode
import copy 

from modules.sumo import ErrorFunctionConnexionVersion,TLIncomingCorridors,CompleteCorridorsConnexions,ConnexionsGreenTime,connexions_routes_correspondance_matrix,update_green_time_dict,update_green_times,SumoSimulation
from modules.gp import UpdatedFAFunctionLinearRegresion,gp_initialization
from modules.bo import InitialSamples,Optimization,BOLoop

c:\Users\sh032\anaconda3\envs\thesis\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Creation of the Ground Truth dataframe

In [ ]:
!python gt_dataset_generation_script.py

# Creation of the initial sample

In [4]:
additional_xml = "ingolstadt7/additional.xml"
x_dim=7
gt_dir = 'gt_data/'
output_dir = 'outputs/'
init_output_dir="initial_output/"

scaling_loss_factor=1

num_replications = 5

def get_tls():
    green_phase_times_dict_tmp = update_green_time_dict(33*np.ones(x_dim))
    tl_ids_set = []
    for elem in green_phase_times_dict_tmp.keys():
        if elem[0] not in tl_ids_set:
            tl_ids_set.append(elem[0])
    return tl_ids_set   

tl_ids = get_tls()
print(tl_ids)

df_gt = pd.read_csv(f'./gt_data/df_gt_InTAS7DIM.csv')
print(f'Number of GT connexions: {df_gt.shape[0]}')


low_bound = 5
up_bound = 74
device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
dtype = torch.double

x_bounds = torch.tensor([[low_bound]*x_dim,
                    [up_bound]*(x_dim-1)+[50]], 
                    device=device, dtype=dtype)

cfg_file="ingolstadt7/ingolstadt7.sumocfg"

sumo_simulator=SumoSimulation(
    cfg_file=cfg_file,
    output_dir=init_output_dir,
    tl_ids=tl_ids_set
)


net_xml="./ingolstadt7/ingolstadt7.net.xml"
route_xml="./ingolstadt7/ingolstadt7_routes_with_edges.rou.xml"
corridors_generator=TLIncomingCorridors(net_xml,route_xml)
corridors_df=corridors_generator.get_corridors_df()

dtype=torch.double



corridors_generator=TLIncomingCorridors(net_xml,route_xml)
corridors_df=corridors_generator.get_corridors_df()

g_dict_to_vary={
        "gneJ260":"GGGGGgrrr",
        "gneJ210": "GGggrrrrrrGGGG",
        "32564122": "GGGGGgrrr",
        "gneJ207":"GGgGrGGG",
        "gneJ143": "rrrGGGGgGGGg",
        "cluster_1757124350_1757124352":"GGgrrGGG",
        "cluster_306484187_cluster_1200363791_1200363826_1200363834_1200363898_1200363927_1200363938_1200363947_1200364074_1200364103_1507566554_1507566556_255882157_306484190":"GGGGGGrrrrrr"
    }

g_dict_to_change={
        "gneJ260":"GrrrrrGGG",
        "gneJ210": "rrrrGGGGGGGGrr",
        "32564122": "GrrrrrGGG",
        "gneJ207":"rrrGGGrr",
        "gneJ143": "GGGGrrrrrrrr",
        "cluster_1757124350_1757124352":"rrrGGGrr",
        "cluster_306484187_cluster_1200363791_1200363826_1200363834_1200363898_1200363927_1200363938_1200363947_1200364074_1200364103_1507566554_1507566556_255882157_306484190":"rrrrrrrrGGGG"
    }

g_dict_const={
    "gneJ260":["yyyyygrrr","rrrGGGrrr","rrryyyrrr","yrrrrryyy"],
    "gneJ210": ["yyggrrrrrryyyy","GGGGrrrrrrrrrr","yyyyrrrrrrrrrr","rrrryyyyyyyyrr"],
    "32564122": ["yyyyyyrrr","yrrrrryyy"],
    "gneJ207":["yygyryyy","GGGrrrrr","yyyrrrrr","rrryyyrr"],
    "gneJ143": ["rrryyyygyyyg","rrrrrrrGrrrG","rrrrrrryrrry","yyyyrrrrrrrr"],
    "cluster_1757124350_1757124352":["yygrryyy","GGGrrrrr","yyyrrrrr","rrryyyrr"],
    "cluster_306484187_cluster_1200363791_1200363826_1200363834_1200363898_1200363927_1200363938_1200363947_1200364074_1200364103_1507566554_1507566556_255882157_306484190":["rrrrrrrrGGyy","rrrrrrGGGGrr","rrrrGGGGGGrr","rrrrGGyyyyrr","yyyyyyrrrrrr"]
}


complete_corridors_connexions_generator=CompleteCorridorsConnexions(route_xml,net_xml,corridors_df)

complete_corridors_connexions_df=complete_corridors_connexions_generator.create_connexion_corridors_df()

connexions_routes_correspondance_tensor,connexions_routes_correspondance_df=connexions_routes_correspondance_matrix(complete_corridors_connexions_df)



error_function=ErrorFunctionConnexionVersion(scaling_factor_loss_fn=scaling_loss_factor,
                                         df_gt=df_gt,
                                         sumo_simulator=sumo_simulator,
                                         complete_corridors_connexions_df=complete_corridors_connexions_df,
                                         additional_xml=additional_xml,
                                         output_dir=init_output_dir,
                                         tl_ids=tl_ids)

initial_sampler=InitialSamples(
    objective_function=error_function,
    dtype=dtype
)

init_sample_x_norm = []
init_sample_y = []
init_df = []

num_restarts=3
init_sample_size=5

for id_init_sample in range(num_restarts):
    x_sample, y_sample, df_init = initial_sampler.get_initial_sample(sample_size=init_sample_size,
                                                                     x_dim=x_dim,
                                                                     x_bounds=x_bounds,
                                                                     seed=id_init_sample,
                                                                     id_init_sample=id_init_sample,
                                                                     num_replications=num_replications)
    init_sample_x_norm.append(x_sample)
    init_sample_y.append(y_sample)
    init_df.append(df_init)

torch.save(init_sample_x_norm,"./initial_samples/init_sample_x_norm_route_error_route_file.pt")
torch.save(init_sample_y,"./initial_samples/init_sample_y_route_error_route_file.pt")
torch.save(init_df,"./initial_samples/init_df_route_error_route_file.pt")

['gneJ260', 'gneJ210', '32564122', 'gneJ207', 'gneJ143', 'cluster_1757124350_1757124352', 'cluster_306484187_cluster_1200363791_1200363826_1200363834_1200363898_1200363927_1200363938_1200363947_1200364074_1200364103_1507566554_1507566556_255882157_306484190']
Number of GT connexions: 81
G_ALL :
[37, 38, 45, 30, 39, 45, 26, 49, 70, 5, 10, 65, 9, 42]
G_ALL :
[59, 16, 27, 48, 54, 30, 51, 24, 15, 60, 70, 5, 43, 8]
G_ALL :
[49, 26, 72, 3, 5, 79, 15, 60, 30, 45, 51, 24, 16, 35]
G_ALL :
[11, 64, 22, 53, 58, 26, 63, 12, 43, 32, 26, 49, 27, 24]
G_ALL :
[17, 58, 60, 15, 43, 41, 65, 10, 49, 26, 17, 58, 24, 27]
Sample number 0
00_initial_sample_0


C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)
C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)
C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)
C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)
C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)


*** current green times:  [37.78239632 45.88415545 39.11906543 26.65577033 70.97033566 10.74785436
  9.44435697]
Sample number 1
01_initial_sample_0


C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)
C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)
C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)
C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)
C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)


*** current green times:  [59.05137557 27.17015651 54.97671831 51.98866647 15.73605017 70.01568687
 43.6641717 ]
Sample number 2
02_initial_sample_0


C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)
C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)
C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)
C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)
C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)


... (13 outputs left out)

C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)
C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)
C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)
C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)
C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)


*** current green times:  [10.2985124  14.35232742 14.10373917 43.03536445 58.85360307 23.53799713
 28.4685564 ]
<class 'torch.Tensor'>
 x dimention of initial sample torch.Size([5, 7])
G_ALL :
[58, 17, 71, 4, 6, 78, 49, 26, 71, 4, 21, 54, 37, 14]
G_ALL :
[9, 66, 18, 57, 59, 25, 26, 49, 34, 41, 59, 16, 25, 26]
G_ALL :
[37, 38, 48, 27, 31, 53, 70, 5, 19, 56, 49, 26, 45, 6]
G_ALL :
[48, 27, 33, 42, 52, 32, 12, 63, 51, 24, 28, 47, 12, 39]
G_ALL :
[44, 31, 41, 34, 72, 12, 21, 54, 43, 32, 11, 64, 43, 8]
Sample number 0
00_initial_sample_2


C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)
C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)
C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)
C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)
C:\Program Files (x86)\Eclipse\Sumo\tools\traci\main.py:200: UserWarning: API change now handles step as floating point seconds
  connection.check().simulationStep(step)


FatalTraCIError: Connection closed by SUMO.